In [ ]:
library("DESeq2")
library(apeglm)

# Read in data

**Sample metadata**

In [117]:
sample_map <- read.table("../rna_seq/samples.csv", sep=",", header=FALSE,
 col.names=c("SampleID", "path", 'genome', 'strain', 'Treatment'))
# add GC/AC sialic column
sample_map$sialic <- lapply(sample_map$Treatment, function(x) grepl("C", x))
sample_map$sialic <- factor(as.integer(sample_map$sialic))
sample_map$Treatment <- factor(sample_map$Treatment)
sample_map$Treatment <- factor(
  sample_map$Treatment,
  levels = c("Glucose", "AC", "GC")   # Glucose = reference
)
head(sample_map)

,SampleID,path,genome,strain,Treatment,sialic
,<chr>,<chr>,<chr>,<chr>,<fct>,<fct>
1,B01,vulgatus_samples/B01_S74_R1_001.fastq.gz,CL09,CL09,Glucose,0
2,F01,vulgatus_samples/F01_S78_R1_001.fastq.gz,CL09,CL09,Glucose,0
3,G01,vulgatus_samples/G01_S79_R1_001.fastq.gz,CL09,CL09,Glucose,0
4,E02,vulgatus_samples/E02_S85_R1_001.fastq.gz,CL09,CL09,AC,1
5,F02,vulgatus_samples/F02_S86_R1_001.fastq.gz,CL09,CL09,AC,1
6,G02,vulgatus_samples/G02_S87_R1_001.fastq.gz,CL09,CL09,AC,1


**counts** (double bc fadu half counts single reads)

In [76]:
load_counts <- function(samples, count_dir = "../rna_seq/vulgatus_processed") {
  counts <- Reduce(
    function(x, y) merge(x, y, by = "featureID", sort = FALSE),
    lapply(samples, function(sample) {
      df <- read.table(
        file.path(count_dir, sample, paste0(sample, ".mapped.counts.txt")),
        header = TRUE, sep = "\t"
      )
      out <- data.frame(featureID = df$featureID, counts = df$counts * 2)
      names(out)[2] <- sample
      out
    })
  )
  rownames(counts) <- counts$featureID
  counts$featureID <- NULL
  counts
}

CL09s <- subset(sample_map, strain == "CL09")$SampleID
ATCCs <- subset(sample_map, strain == "ATCC")$SampleID
BIOMLs <- subset(sample_map, strain == "BIOML")$SampleID

CL09_counts <- load_counts(CL09s)
ATCC_counts <- load_counts(ATCCs)
#BIOML_counts <- load_counts(BIOMLs)

#head(CL09_counts)

In [77]:
dim(CL09_counts)
dim(ATCC_counts)
dim(BIOML_counts)

[1] 4025    9

[1] 4194    8

ERROR: Error: object 'BIOML_counts' not found


# Expression per Treatment, by strain

## ATCC (expect to see nan genes come on in sialic acid)

**subset metadata, filter out low count genes**

In [120]:
meta_subset <- subset(sample_map, strain == "ATCC")
row.names(meta_subset) <- meta_subset$SampleID
media <- meta_subset$Treatment
tf <- rowSums(ATCC_counts > 40) >= 3
ATCC_counts.adj <- round(ATCC_counts[tf, , drop = FALSE])

**run deseq**

In [121]:
dds_ATCC_sialic <- DESeqDataSetFromMatrix(countData = ATCC_counts.adj,
 colData = meta_subset, design = ~ sialic)

dds_ATCC_treatment <- DESeqDataSetFromMatrix(countData = ATCC_counts.adj,
 colData = meta_subset, design = ~ Treatment)

dds_ATCC_sialic_deseq <- DESeq(dds_ATCC_sialic)
dds_ATCC_treatment_deseq <- DESeq(dds_ATCC_treatment)

ATCC_sialic_res <- results(dds_ATCC_sialic_deseq, alpha = 0.05) 
ATCC_treatment_res <- results(dds_ATCC_treatment_deseq, alpha = 0.05) 

converting counts to integer mode

converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



**shrink coefficients with apeglm**

In [130]:
ATCC_sialic_shrink <- lfcShrink(dds_ATCC_sialic_deseq, coef="sialic_1_vs_0")

## and save a tsv 
write.table(
  as.data.frame(ATCC_sialic_shrink),
  file = "results/atcc.sialic.tsv",
  sep = "\t",
  quote = FALSE,
  row.names = TRUE
)

using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895



## CL09 (expect mystery transporter)

**subset metadata, filter out low-count genes (less than 40 reads in 3 samples)**

In [131]:
meta_subset <- subset(sample_map, strain == "CL09")
row.names(meta_subset) <- meta_subset$SampleID
media <- meta_subset$Treatment
tf <- rowSums(CL09_counts > 40) >= 3
CL09_counts.adj <- round(CL09_counts[tf, , drop = FALSE])

**run DESeq**

In [132]:
dds_CL09_sialic <- DESeqDataSetFromMatrix(countData = CL09_counts.adj,
 colData = meta_subset, design = ~ sialic)

dds_CL09_treatment <- DESeqDataSetFromMatrix(countData = CL09_counts.adj,
 colData = meta_subset, design = ~ Treatment)

dds_CL09_sialic_deseq <- DESeq(dds_CL09_sialic)
dds_CL09_treatment_deseq <- DESeq(dds_CL09_treatment)

CL09_sialic_res <- results(dds_CL09_sialic_deseq, alpha = 0.05) 
CL09_treatment_res <- results(dds_CL09_treatment_deseq, alpha = 0.05) 

converting counts to integer mode

converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [133]:
CL09_sialic_shrink <- lfcShrink(dds_CL09_sialic_deseq, coef="sialic_1_vs_0")

## and save a tsv 
write.table(
  as.data.frame(CL09_sialic_shrink),
  file = "results/cl09.sialic.tsv",
  sep = "\t",
  quote = FALSE,
  row.names = TRUE
)

using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895

